In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [5]:
matches= pd.read_csv(r"D:\Projects\15\5.IPL_Winner\data\matches.csv")
deliveries= pd.read_csv(r"D:\Projects\15\5.IPL_Winner\data\deliveries.csv")

In [6]:
print(f"Matches: {matches.shape}")
print(f"Deliveries: {deliveries.shape}")
print(f"\nMatches columns:\n{matches.columns.tolist()}")
print(f"\nDeliveries columns:\n{deliveries.columns.tolist()}")

Matches: (1095, 20)
Deliveries: (260920, 17)

Matches columns:
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

Deliveries columns:
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


In [7]:
# Checking match types
print("Match types:")
print(matches['match_type'].value_counts())

print("\nSeasons covered:")
print(sorted(matches['season'].unique()))

print("\nTotal teams:")
print(sorted(matches['team1'].unique()))

print("\nMissing values in matches:")
print(matches.isnull().sum())

print("\nWinner sample:")
print(matches['winner'].value_counts().head(10))

Match types:
match_type
League                1029
Final                   17
Qualifier 2             14
Qualifier 1             14
Eliminator              11
Semi Final               6
Elimination Final        3
3rd Place Play-Off       1
Name: count, dtype: int64

Seasons covered:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024']

Total teams:
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

Missing values in matches:
id                    0
season                0
city                 51
date                  0
match_type     

In [8]:
# Fixing the team name inconsistencies
team_name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Deccan Chargers': 'Sunrisers Hyderabad'
}

# Apply to matches
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(team_name_map)

# Apply to deliveries
for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(team_name_map)

# Fix season format
matches['season'] = matches['season'].str[:4].astype(int)

# Remove matches with no result (rain, etc.)
matches = matches[matches['winner'].notna()]
matches = matches[matches['result'] != 'no result']

print(f"Matches after cleaning: {matches.shape}")
print(f"\nUnique teams now: {sorted(matches['team1'].unique())}")
print(f"\nSeasons now: {sorted(matches['season'].unique())}")

Matches after cleaning: (1090, 20)

Unique teams now: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

Seasons now: [np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [9]:
# ── BATTING STATS FILTERED ──
batting_stats = deliveries.groupby('batter').agg(
    total_runs=('batsman_runs', 'sum'),
    balls_faced=('batsman_runs', 'count'),
    innings=('match_id', 'nunique')
).reset_index()

batting_stats['strike_rate'] = (batting_stats['total_runs'] / batting_stats['balls_faced'] * 100).round(2)
batting_stats['avg_runs_per_match'] = (batting_stats['total_runs'] / batting_stats['innings']).round(2)

# Filtering players with minimum 100 balls faced
batting_stats = batting_stats[batting_stats['balls_faced'] >= 100]

print(f"Batters with 100+ balls: {len(batting_stats)}")
print("\nTop 10 batters by strike rate:")
print(batting_stats.nlargest(10, 'strike_rate')[['batter', 'strike_rate', 'total_runs', 'innings']])

Batters with 100+ balls: 300

Top 10 batters by strike rate:
              batter  strike_rate  total_runs  innings
234  J Fraser-McGurk       220.00         330        9
652         WG Jacks       172.93         230        8
433          PD Salt       169.61         653       21
606         T Stubbs       169.46         405       17
617          TM Head       168.56         772       25
39        AD Russell       164.22        2488      104
105      BCJ Cutting       163.01         238       17
208        H Klaasen       161.99         993       32
503  Ramandeep Singh       160.38         170       13
85   Ashutosh Sharma       160.17         189        9


In [10]:
# ── BOWLING STATS FILTERED ──
bowling_stats = deliveries.groupby('bowler').agg(
    runs_conceded=('total_runs', 'sum'),
    balls_bowled=('total_runs', 'count'),
    wickets=('is_wicket', 'sum'),
    matches=('match_id', 'nunique')
).reset_index()

bowling_stats['economy'] = (bowling_stats['runs_conceded'] / bowling_stats['balls_bowled'] * 6).round(2)
bowling_stats['bowling_avg'] = (bowling_stats['runs_conceded'] / bowling_stats['wickets'].replace(0, np.nan)).round(2)
bowling_stats['bowling_sr'] = (bowling_stats['balls_bowled'] / bowling_stats['wickets'].replace(0, np.nan)).round(2)

# Filtering bowlers with minimum 120 balls bowled (20 overs)
bowling_stats = bowling_stats[bowling_stats['balls_bowled'] >= 120]

print(f"Bowlers with 120+ balls: {len(bowling_stats)}")
print("\nTop 10 bowlers by economy:")
print(bowling_stats.nsmallest(10, 'economy')[['bowler', 'economy', 'wickets', 'matches']])

Bowlers with 120+ balls: 319

Top 10 bowlers by economy:
              bowler  economy  wickets  matches
473    Sohail Tanvir     6.23       24       11
2         A Chandila     6.28       11       12
141       FH Edwards     6.40        6        6
443  SMSM Senanayake     6.49        9        8
442       SM Pollock     6.58       13       13
7           A Kumble     6.65       49       42
147       GD McGrath     6.67       14       14
263   M Muralitharan     6.70       67       66
173         IS Sodhi     6.73        9        8
181          J Yadav     6.74        9       20


In [11]:
# ── TEAM STRENGTH FUNCTION ──
def get_team_batting_strength(players):
    scores = []
    for player in players:
        if player in batting_stats['batter'].values:
            row = batting_stats[batting_stats['batter'] == player].iloc[0]
            # Weighted score: 60% strike rate + 40% avg runs per match
            score = (row['strike_rate'] * 0.6) + (row['avg_runs_per_match'] * 0.4)
            scores.append(score)
    return np.mean(scores) if scores else 100  # default if no stats

def get_team_bowling_strength(players):
    scores = []
    for player in players:
        if player in bowling_stats['bowler'].values:
            row = bowling_stats[bowling_stats['bowler'] == player].iloc[0]
            # Lower economy = better. Invert so higher = better
            score = (1 / row['economy']) * 100
            # Add wicket taking ability
            if not np.isnan(row['bowling_sr']):
                score += (1 / row['bowling_sr']) * 50
            scores.append(score)
    return np.mean(scores) if scores else 10  # default if no stats

# Test with sample players
mi_batters = ['RG Sharma', 'Q de Kock', 'SR Tendulkar', 'KA Pollard', 'HH Pandya']
csk_bowlers = ['R Jadeja', 'DJ Bravo', 'Imran Tahir', 'DP Nannes', 'MM Sharma']

print(f"MI Batting Strength: {get_team_batting_strength(mi_batters):.2f}")
print(f"CSK Bowling Strength: {get_team_bowling_strength(csk_bowlers):.2f}")

MI Batting Strength: 88.50
CSK Bowling Strength: 15.85


In [12]:
# Loading playing XI data from deliveries
def get_playing_xi(match_id, team):
    match_deliveries = deliveries[deliveries['match_id'] == match_id]
    batters = match_deliveries[match_deliveries['batting_team'] == team]['batter'].unique().tolist()
    bowlers = match_deliveries[match_deliveries['bowling_team'] == team]['bowler'].unique().tolist()
    players = list(set(batters + bowlers))
    return players

# Build features for each match
print("Building training dataset...")
rows = []

for _, match in matches.iterrows():
    try:
        team1 = match['team1']
        team2 = match['team2']
        winner = match['winner']
        toss_winner = match['toss_winner']
        toss_decision = match['toss_decision']
        venue = match['venue']
        match_id = match['id']

        # Get playing XI
        team1_players = get_playing_xi(match_id, team1)
        team2_players = get_playing_xi(match_id, team2)

        # Calculate strengths
        team1_bat = get_team_batting_strength(team1_players)
        team1_bowl = get_team_bowling_strength(team1_players)
        team2_bat = get_team_batting_strength(team2_players)
        team2_bowl = get_team_bowling_strength(team2_players)

        # Toss advantage
        toss_team1 = 1 if toss_winner == team1 else 0
        bat_first = 1 if toss_decision == 'bat' else 0

        # Target: 1 if team1 wins, 0 if team2 wins
        target = 1 if winner == team1 else 0

        rows.append({
            'team1': team1,
            'team2': team2,
            'venue': venue,
            'team1_bat_strength': team1_bat,
            'team1_bowl_strength': team1_bowl,
            'team2_bat_strength': team2_bat,
            'team2_bowl_strength': team2_bowl,
            'toss_team1': toss_team1,
            'bat_first': bat_first,
            'target': target
        })
    except:
        continue

df = pd.DataFrame(rows)
print(f"Training dataset shape: {df.shape}")
print(f"\nSample:")
print(df.head())
print(f"\nTarget distribution:")
print(df['target'].value_counts(normalize=True).round(3))

Building training dataset...
Training dataset shape: (1090, 10)

Sample:
                         team1                        team2  \
0  Royal Challengers Bengaluru        Kolkata Knight Riders   
1                 Punjab Kings          Chennai Super Kings   
2               Delhi Capitals             Rajasthan Royals   
3               Mumbai Indians  Royal Challengers Bengaluru   
4        Kolkata Knight Riders          Sunrisers Hyderabad   

                                        venue  team1_bat_strength  \
0                       M Chinnaswamy Stadium           73.970250   
1  Punjab Cricket Association Stadium, Mohali           77.049250   
2                            Feroz Shah Kotla           83.000667   
3                            Wankhede Stadium           78.902667   
4                                Eden Gardens           72.037500   

   team1_bowl_strength  team2_bat_strength  team2_bowl_strength  toss_team1  \
0            14.698282           71.387000            

In [13]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

# Encode venue
le_venue = LabelEncoder()
df['venue_encoded'] = le_venue.fit_transform(df['venue'])

# Encode teams
le_team = LabelEncoder()
all_teams = pd.concat([df['team1'], df['team2']]).unique()
le_team.fit(all_teams)
df['team1_encoded'] = le_team.transform(df['team1'])
df['team2_encoded'] = le_team.transform(df['team2'])

# Features
feature_cols = [
    'team1_encoded', 'team2_encoded', 'venue_encoded',
    'team1_bat_strength', 'team1_bowl_strength',
    'team2_bat_strength', 'team2_bowl_strength',
    'toss_team1', 'bat_first'
]

X = df[feature_cols]
y = df['target']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

print("Training models...\n")
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5)
    results[name] = {
        'Test Accuracy': round(acc, 4),
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4)
    }
    print(f"{name}: Test={acc:.4f} | CV={cv_scores.mean():.4f} (+/-{cv_scores.std():.4f})")

print("\n--- Results Summary ---")
print(pd.DataFrame(results).T)

Training models...

Random Forest: Test=0.4633 | CV=0.5284 (+/-0.0262)
Gradient Boosting: Test=0.5550 | CV=0.5128 (+/-0.0160)
Logistic Regression: Test=0.5138 | CV=0.5239 (+/-0.0303)

--- Results Summary ---
                     Test Accuracy  CV Mean  CV Std
Random Forest               0.4633   0.5284  0.0262
Gradient Boosting           0.5550   0.5128  0.0160
Logistic Regression         0.5138   0.5239  0.0303


In [14]:
# ── HEAD TO HEAD WIN RATE ──
def get_h2h_winrate(team1, team2, before_season=None):
    h2h = matches[(
        ((matches['team1'] == team1) & (matches['team2'] == team2)) |
        ((matches['team1'] == team2) & (matches['team2'] == team1))
    )]
    if before_season:
        h2h = h2h[h2h['season'] < before_season]
    if len(h2h) == 0:
        return 0.5
    wins = len(h2h[h2h['winner'] == team1])
    return round(wins / len(h2h), 3)

# ── VENUE WIN RATE PER TEAM ──
def get_venue_winrate(team, venue, before_season=None):
    venue_matches = matches[
        ((matches['team1'] == team) | (matches['team2'] == team)) &
        (matches['venue'] == venue)
    ]
    if before_season:
        venue_matches = venue_matches[venue_matches['season'] < before_season]
    if len(venue_matches) == 0:
        return 0.5
    wins = len(venue_matches[venue_matches['winner'] == team])
    return round(wins / len(venue_matches), 3)

# ── RECENT FORM (last 5 matches win rate) ──
def get_recent_form(team, before_date, n=5):
    team_matches = matches[
        ((matches['team1'] == team) | (matches['team2'] == team)) &
        (matches['date'] < before_date)
    ].tail(n)
    if len(team_matches) == 0:
        return 0.5
    wins = len(team_matches[team_matches['winner'] == team])
    return round(wins / len(team_matches), 3)

# Test
print(f"MI vs CSK H2H winrate: {get_h2h_winrate('Mumbai Indians', 'Chennai Super Kings')}")
print(f"MI at Wankhede winrate: {get_venue_winrate('Mumbai Indians', 'Wankhede Stadium')}")

MI vs CSK H2H winrate: 0.541
MI at Wankhede winrate: 0.627


In [15]:
# Converting date to string for comparison
matches['date'] = matches['date'].astype(str)

print("Rebuilding dataset with enhanced features...")
rows = []

for _, match in matches.iterrows():
    try:
        team1 = match['team1']
        team2 = match['team2']
        winner = match['winner']
        toss_winner = match['toss_winner']
        toss_decision = match['toss_decision']
        venue = match['venue']
        match_id = match['id']
        season = match['season']
        date = match['date']

        # Get playing XI
        team1_players = get_playing_xi(match_id, team1)
        team2_players = get_playing_xi(match_id, team2)

        # Player strength features
        team1_bat = get_team_batting_strength(team1_players)
        team1_bowl = get_team_bowling_strength(team1_players)
        team2_bat = get_team_batting_strength(team2_players)
        team2_bowl = get_team_bowling_strength(team2_players)

        # New features
        h2h = get_h2h_winrate(team1, team2, before_season=season)
        team1_venue = get_venue_winrate(team1, venue, before_season=season)
        team2_venue = get_venue_winrate(team2, venue, before_season=season)
        team1_form = get_recent_form(team1, date)
        team2_form = get_recent_form(team2, date)

        # Toss features
        toss_team1 = 1 if toss_winner == team1 else 0
        bat_first = 1 if toss_decision == 'bat' else 0

        # Target
        target = 1 if winner == team1 else 0

        rows.append({
            'team1': team1,
            'team2': team2,
            'venue': venue,
            'team1_bat_strength': team1_bat,
            'team1_bowl_strength': team1_bowl,
            'team2_bat_strength': team2_bat,
            'team2_bowl_strength': team2_bowl,
            'h2h_winrate': h2h,
            'team1_venue_winrate': team1_venue,
            'team2_venue_winrate': team2_venue,
            'team1_form': team1_form,
            'team2_form': team2_form,
            'toss_team1': toss_team1,
            'bat_first': bat_first,
            'target': target
        })
    except:
        continue

df2 = pd.DataFrame(rows)
print(f"Enhanced dataset shape: {df2.shape}")
print(f"Features: {df2.columns.tolist()}")

Rebuilding dataset with enhanced features...
Enhanced dataset shape: (1090, 15)
Features: ['team1', 'team2', 'venue', 'team1_bat_strength', 'team1_bowl_strength', 'team2_bat_strength', 'team2_bowl_strength', 'h2h_winrate', 'team1_venue_winrate', 'team2_venue_winrate', 'team1_form', 'team2_form', 'toss_team1', 'bat_first', 'target']


In [16]:
# Encoding categorical columns
df2['venue_encoded'] = le_venue.fit_transform(df2['venue'])
df2['team1_encoded'] = le_team.transform(df2['team1'])
df2['team2_encoded'] = le_team.transform(df2['team2'])

# Updated feature columns
feature_cols2 = [
    'team1_encoded', 'team2_encoded', 'venue_encoded',
    'team1_bat_strength', 'team1_bowl_strength',
    'team2_bat_strength', 'team2_bowl_strength',
    'h2h_winrate', 'team1_venue_winrate', 'team2_venue_winrate',
    'team1_form', 'team2_form',
    'toss_team1', 'bat_first'
]

X2 = df2[feature_cols2]
y2 = df2['target']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print("Retraining with enhanced features...\n")
results2 = {}
for name, model in models.items():
    model.fit(X2_train, y2_train)
    y_pred = model.predict(X2_test)
    acc = accuracy_score(y2_test, y_pred)
    cv_scores = cross_val_score(model, X2, y2, cv=5)
    results2[name] = {
        'Test Accuracy': round(acc, 4),
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4)
    }
    print(f"{name}: Test={acc:.4f} | CV={cv_scores.mean():.4f} (+/-{cv_scores.std():.4f})")

print("\n--- Enhanced Results ---")
print(pd.DataFrame(results2).T)

Retraining with enhanced features...

Random Forest: Test=0.4725 | CV=0.4908 (+/-0.0308)
Gradient Boosting: Test=0.4954 | CV=0.4862 (+/-0.0359)
Logistic Regression: Test=0.5229 | CV=0.5138 (+/-0.0500)

--- Enhanced Results ---
                     Test Accuracy  CV Mean  CV Std
Random Forest               0.4725   0.4908  0.0308
Gradient Boosting           0.4954   0.4862  0.0359
Logistic Regression         0.5229   0.5138  0.0500


In [17]:
import joblib
import json
import os

os.makedirs('../models', exist_ok=True)

# Save best model (Logistic Regression had most consistent CV)
best_model = models['Logistic Regression']
joblib.dump(best_model, '../models/ipl_model.pkl')

# Save encoders
joblib.dump(le_venue, '../models/le_venue.pkl')
joblib.dump(le_team, '../models/le_team.pkl')

# Save batting and bowling stats
batting_stats.to_csv('../models/batting_stats.csv', index=False)
bowling_stats.to_csv('../models/bowling_stats.csv', index=False)

# Save feature columns
with open('../models/feature_cols.json', 'w') as f:
    json.dump(feature_cols2, f)

# Save all team names
all_teams_list = sorted(df2['team1'].unique().tolist())
with open('../models/teams.json', 'w') as f:
    json.dump(all_teams_list, f)

# Save all venues
all_venues_list = sorted(df2['venue'].unique().tolist())
with open('../models/venues.json', 'w') as f:
    json.dump(all_venues_list, f)

print("All models and data saved!")
print(f"Teams: {all_teams_list}")

All models and data saved!
Teams: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
